# Pnkln Judge 6 - Development Workbench

**Purpose:** Develop and test Judge 6 multi-LLM inference system  
**SLA Target:** p99 ≤90ms, p95 ≤50ms, p50 ≤20ms  
**Architecture:** Hybrid enforcement (Gemini 40%, Claude 35%, GPT-5 15%, Grok 5%, Mistral 5%)

---

## 1. Setup & Environment

In [ ]:
# Install required packages
!pip install --quiet --upgrade \
    anthropic \
    openai \
    google-generativeai \
    torch \
    transformers \
    langchain \
    langchain-anthropic \
    langchain-openai \
    langchain-google-genai \
    fastapi \
    uvicorn \
    pydantic \
    prometheus-client \
    kubernetes \
    google-cloud-secret-manager \
    google-cloud-monitoring

In [ ]:
import os
import time
from dataclasses import dataclass
from enum import Enum

# LLM Clients
import numpy as np
from google.cloud import secretmanager

# LangChain
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

print("✓ Imports successful")

## 2. Configuration & API Keys

In [ ]:
# GCP Configuration
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
REGION = "us-central1"

# SLA Targets (milliseconds)
TARGET_P99 = 90
TARGET_P95 = 50
TARGET_P50 = 20

# LLM Weights (Judge 6 Hybrid)
LLM_WEIGHTS = {"gemini": 0.40, "claude": 0.35, "gpt5": 0.15, "grok": 0.05, "mistral": 0.05}

print(f"Project: {PROJECT_ID}")
print(f"SLA Targets: p99≤{TARGET_P99}ms, p95≤{TARGET_P95}ms, p50≤{TARGET_P50}ms")
print(f"LLM Weights: {LLM_WEIGHTS}")

In [ ]:
def get_secret(secret_id: str) -> str:
    """Fetch secret from Secret Manager"""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT_ID}/secrets/{secret_id}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")


# Load API keys from Secret Manager
try:
    ANTHROPIC_API_KEY = get_secret("anthropic-api-key")
    OPENAI_API_KEY = get_secret("openai-api-key")
    XAI_API_KEY = get_secret("xai-api-key")
    MISTRAL_API_KEY = get_secret("mistral-api-key")
    GOOGLE_AI_API_KEY = get_secret("google-ai-api-key")
    print("✓ API keys loaded from Secret Manager")
except Exception as e:
    print(f"⚠ Warning: Could not load secrets from Secret Manager: {e}")
    print("Using environment variables as fallback...")
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    XAI_API_KEY = os.environ.get("XAI_API_KEY", "")
    MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "")
    GOOGLE_AI_API_KEY = os.environ.get("GOOGLE_AI_API_KEY", "")

## 3. Judge 6 Implementation

In [ ]:
class LLMProvider(Enum):
    GEMINI = "gemini"
    CLAUDE = "claude"
    GPT5 = "gpt5"
    GROK = "grok"
    MISTRAL = "mistral"


@dataclass
class InferenceRequest:
    prompt: str
    max_tokens: int = 1000
    temperature: float = 0.7


@dataclass
class InferenceResponse:
    text: str
    provider: LLMProvider
    latency_ms: float
    success: bool
    error: str | None = None

In [ ]:
class CircuitBreaker:
    """Circuit breaker for LLM fallback"""

    def __init__(self, threshold: int = 5, timeout: int = 60):
        self.threshold = threshold
        self.timeout = timeout
        self.failures = {}
        self.last_failure_time = {}

    def record_failure(self, provider: LLMProvider):
        """Record a failure for a provider"""
        if provider not in self.failures:
            self.failures[provider] = 0
        self.failures[provider] += 1
        self.last_failure_time[provider] = time.time()

    def record_success(self, provider: LLMProvider):
        """Record a success for a provider"""
        self.failures[provider] = 0

    def is_available(self, provider: LLMProvider) -> bool:
        """Check if provider is available"""
        if provider not in self.failures:
            return True

        # Reset after timeout
        if time.time() - self.last_failure_time.get(provider, 0) > self.timeout:
            self.failures[provider] = 0
            return True

        return self.failures[provider] < self.threshold

In [ ]:
class Claude_Code_6:
    """Judge 6 Multi-LLM Inference Engine"""

    def __init__(self, weights: dict[str, float]):
        self.weights = weights
        self.circuit_breaker = CircuitBreaker()
        self.latencies = []

        # Initialize LLM clients
        self.clients = {
            LLMProvider.GEMINI: ChatGoogleGenerativeAI(
                model="gemini-pro", google_api_key=GOOGLE_AI_API_KEY
            )
            if GOOGLE_AI_API_KEY
            else None,
            LLMProvider.CLAUDE: ChatAnthropic(
                model="claude-3-5-sonnet-20241022", anthropic_api_key=ANTHROPIC_API_KEY
            )
            if ANTHROPIC_API_KEY
            else None,
            LLMProvider.GPT5: ChatOpenAI(
                model="gpt-4",  # Update to GPT-5 when available
                openai_api_key=OPENAI_API_KEY,
            )
            if OPENAI_API_KEY
            else None,
        }

    def select_provider(self) -> LLMProvider:
        """Select provider using weighted round-robin"""
        # Filter available providers
        available = [
            (provider, weight)
            for provider, weight in self.weights.items()
            if self.circuit_breaker.is_available(LLMProvider(provider))
        ]

        if not available:
            raise Exception("No LLM providers available")

        # Weighted random selection
        providers, weights = zip(*available, strict=False)
        total = sum(weights)
        normalized_weights = [w / total for w in weights]

        selected = np.random.choice(providers, p=normalized_weights)
        return LLMProvider(selected)

    async def infer(self, request: InferenceRequest) -> InferenceResponse:
        """Execute inference with fallback"""
        start_time = time.time()

        # Try primary provider
        provider = self.select_provider()

        try:
            result = await self._call_provider(provider, request)
            latency_ms = (time.time() - start_time) * 1000

            self.circuit_breaker.record_success(provider)
            self.latencies.append(latency_ms)

            return InferenceResponse(
                text=result, provider=provider, latency_ms=latency_ms, success=True
            )
        except Exception:
            self.circuit_breaker.record_failure(provider)

            # Try fallback
            try:
                fallback_provider = self.select_provider()
                result = await self._call_provider(fallback_provider, request)
                latency_ms = (time.time() - start_time) * 1000

                return InferenceResponse(
                    text=result, provider=fallback_provider, latency_ms=latency_ms, success=True
                )
            except Exception as fallback_error:
                return InferenceResponse(
                    text="",
                    provider=provider,
                    latency_ms=(time.time() - start_time) * 1000,
                    success=False,
                    error=str(fallback_error),
                )

    async def _call_provider(self, provider: LLMProvider, request: InferenceRequest) -> str:
        """Call specific LLM provider"""
        client = self.clients.get(provider)

        if client is None:
            raise Exception(f"Provider {provider} not configured")

        # Use LangChain invoke (sync version for compatibility)
        response = client.invoke(request.prompt)
        return response.content

    def get_latency_stats(self) -> dict[str, float]:
        """Calculate latency percentiles"""
        if not self.latencies:
            return {"p50": 0, "p95": 0, "p99": 0}

        latencies_sorted = sorted(self.latencies)
        n = len(latencies_sorted)

        return {
            "p50": latencies_sorted[int(n * 0.50)],
            "p95": latencies_sorted[int(n * 0.95)],
            "p99": latencies_sorted[int(n * 0.99)],
        }


print("✓ Judge 6 implementation complete")

## 4. Testing & Validation

In [ ]:
# Initialize Judge 6
judge = Claude_Code_6(weights=LLM_WEIGHTS)
print("✓ Judge 6 initialized")

In [ ]:
# Test single inference
test_request = InferenceRequest(
    prompt="What is the capital of France?", max_tokens=50, temperature=0.7
)

response = await judge.infer(test_request)
print(f"Response: {response.text}")
print(f"Provider: {response.provider.value}")
print(f"Latency: {response.latency_ms:.2f}ms")
print(f"Success: {response.success}")

In [ ]:
# Benchmark SLA compliance
import matplotlib.pyplot as plt


async def benchmark(num_requests: int = 100):
    """Run benchmark test"""
    print(f"Running benchmark with {num_requests} requests...")

    test_prompts = [
        "What is 2+2?",
        "Name a color.",
        "What day is today?",
        "Count to 5.",
        "Say hello.",
    ]

    for i in range(num_requests):
        prompt = test_prompts[i % len(test_prompts)]
        request = InferenceRequest(prompt=prompt, max_tokens=20)
        await judge.infer(request)

        if (i + 1) % 10 == 0:
            print(f"Completed {i + 1}/{num_requests} requests")

    # Get stats
    stats = judge.get_latency_stats()
    print("\n" + "=" * 60)
    print("BENCHMARK RESULTS")
    print("=" * 60)
    print(
        f"p50: {stats['p50']:.2f}ms (target: ≤{TARGET_P50}ms) {'✓' if stats['p50'] <= TARGET_P50 else '✗'}"
    )
    print(
        f"p95: {stats['p95']:.2f}ms (target: ≤{TARGET_P95}ms) {'✓' if stats['p95'] <= TARGET_P95 else '✗'}"
    )
    print(
        f"p99: {stats['p99']:.2f}ms (target: ≤{TARGET_P99}ms) {'✓' if stats['p99'] <= TARGET_P99 else '✗'}"
    )
    print("=" * 60)

    # Plot latency distribution
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.hist(judge.latencies, bins=50, edgecolor="black")
    plt.axvline(TARGET_P50, color="g", linestyle="--", label=f"p50 target ({TARGET_P50}ms)")
    plt.axvline(TARGET_P95, color="orange", linestyle="--", label=f"p95 target ({TARGET_P95}ms)")
    plt.axvline(TARGET_P99, color="r", linestyle="--", label=f"p99 target ({TARGET_P99}ms)")
    plt.xlabel("Latency (ms)")
    plt.ylabel("Frequency")
    plt.title("Latency Distribution")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(judge.latencies)
    plt.axhline(TARGET_P99, color="r", linestyle="--", label=f"p99 target ({TARGET_P99}ms)")
    plt.xlabel("Request #")
    plt.ylabel("Latency (ms)")
    plt.title("Latency Over Time")
    plt.legend()

    plt.tight_layout()
    plt.show()


# Run benchmark
await benchmark(num_requests=50)

## 5. GKE Deployment Integration

In [ ]:
# Connect to GKE cluster
from kubernetes import client, config

# Load kubeconfig
try:
    config.load_incluster_config()
    print("✓ Using in-cluster config")
except:
    config.load_kube_config()
    print("✓ Using local kubeconfig")

# List pods in pnkln-inference namespace
v1 = client.CoreV1Api()
pods = v1.list_namespaced_pod(namespace="pnkln-inference")

print("\nPods in pnkln-inference namespace:")
for pod in pods.items:
    print(f"  - {pod.metadata.name}: {pod.status.phase}")

In [ ]:
# Deploy updated Judge 6 image
!kubectl set image deployment/pnkln-judge judge=gcr.io/{PROJECT_ID}/pnkln-judge:latest -n pnkln-inference

In [ ]:
# Check deployment status
!kubectl rollout status deployment/pnkln-judge -n pnkln-inference

## 6. Monitoring & Observability

In [ ]:
# Fetch Prometheus metrics
import requests


def get_prometheus_metrics():
    """Fetch metrics from Prometheus"""
    # Port-forward to Prometheus service
    !kubectl port-forward -n pnkln-inference svc/pnkln-judge 9090:9090 &
    time.sleep(2)

    response = requests.get("http://localhost:9090/metrics")
    return response.text


metrics = get_prometheus_metrics()
print(metrics[:1000])  # Print first 1000 characters

## 7. Cost Optimization Analysis

In [ ]:
def estimate_costs(num_pods: int, gpu_count: int):
    """Estimate infrastructure costs"""
    CPU_NODE_HOURLY = 0.476  # n2-standard-16
    GPU_NODE_HOURLY = 1.09  # L4 GPU
    WORKBENCH_HOURLY = 0.95  # n1-standard-8 + T4

    # Calculate daily costs
    cpu_daily = CPU_NODE_HOURLY * 24
    gpu_daily = GPU_NODE_HOURLY * 24 * gpu_count
    workbench_daily = WORKBENCH_HOURLY * 24

    total_daily = cpu_daily + gpu_daily + workbench_daily
    total_monthly = total_daily * 30

    print("=" * 60)
    print("COST ESTIMATE")
    print("=" * 60)
    print(f"CPU nodes (1):        ${cpu_daily:.2f}/day")
    print(f"GPU nodes ({gpu_count}):        ${gpu_daily:.2f}/day")
    print(f"Workbench (1):        ${workbench_daily:.2f}/day")
    print("-" * 60)
    print(f"Total Daily:          ${total_daily:.2f}")
    print(f"Total Monthly:        ${total_monthly:.2f}")
    print("=" * 60)

    return total_daily, total_monthly


# Estimate for dev environment (1 GPU)
print("\nDevelopment Environment:")
estimate_costs(num_pods=2, gpu_count=1)

# Estimate for prod environment (5 GPUs)
print("\nProduction Environment:")
estimate_costs(num_pods=10, gpu_count=5)

## 8. Next Steps

1. **Fine-tune weights**: Adjust LLM weights based on performance metrics
2. **Implement PyTorch model**: Add custom rules engine for low-latency filtering
3. **Load testing**: Run comprehensive load tests to validate SLA compliance
4. **Cost optimization**: Implement request batching and caching
5. **Monitoring**: Set up Cloud Monitoring dashboards and alerts
6. **Production deployment**: Roll out to production with gradual traffic shifting

---

**Documentation:**  
- [GKE Best Practices](https://cloud.google.com/kubernetes-engine/docs/best-practices)
- [Vertex AI Workbench](https://cloud.google.com/vertex-ai/docs/workbench)
- [Judge 6 Deployment Guide](./README.md)